# 06 – Model Export & Inference Packaging

**Goals**
- Package the best models (XGBoost + Transformer) for production
- Create a unified inference interface
- Export everything needed by the Gradio / FastAPI app
- Verify that loading + prediction works correctly
- No training or data creation – only export & test

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

print("Export started at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Export started at: 2026-10-01 20:31:43


In [2]:
## 1. Paths

ML_DIR   = Path("models/ml")
DL_DIR   = Path("models/dl/best_model")
EXPORT_DIR = Path("models/exported")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("ML model dir :", ML_DIR)
print("DL model dir :", DL_DIR)
print("Export dir   :", EXPORT_DIR)

ML model dir : models\ml
DL model dir : models\dl\best_model
Export dir   : models\exported


In [3]:
## 2. Export Classical ML (XGBoost) bundle

# Load existing artefacts
preprocessor = joblib.load(ML_DIR / "preprocessor.joblib")
xgb_model    = joblib.load(ML_DIR / "xgb_model.joblib")
ml_meta      = joblib.load(ML_DIR / "meta.joblib")

# Create a clean export bundle
ml_bundle = {
    "preprocessor": preprocessor,
    "model": xgb_model,
    "threshold": float(ml_meta.get("threshold", 0.70)),
    "label2id": ml_meta.get("label2id", {"benign": 0, "phish": 1}),
    "id2label": {0: "benign", 1: "phish"},
    "feature_names": [
        "url_len", "domain_len", "path_len", "num_dots", "num_hyphens",
        "num_underscores", "num_digits", "num_params", "has_https",
        "has_ip", "num_subdomains", "text_len", "html_len_raw"
    ],
    "model_type": "xgboost",
    "exported_at": datetime.now().isoformat()
}

joblib.dump(ml_bundle, EXPORT_DIR / "ml_bundle.joblib")
print("✓ Classical ML bundle saved →", EXPORT_DIR / "ml_bundle.joblib")

✓ Classical ML bundle saved → models\exported\ml_bundle.joblib


In [ ]:
## 3. Export Transformer model (already mostly ready)

# The Trainer already saved the model + tokenizer.
# We just copy meta and make sure everything is consistent.

with open(DL_DIR / "meta.json") as f:
    dl_meta = json.load(f)

# Re-save a clean meta file inside export
dl_export_meta = {
    "model_path": str(DL_DIR),
    "model_name": dl_meta.get("model_name", "distilbert-base-uncased"),
    "max_length": dl_meta.get("max_length", 256),
    "threshold": float(dl_meta.get("threshold", 0.70)),
    "label2id": dl_meta.get("label2id", {"benign": 0, "phish": 1}),
    "id2label": {0: "benign", 1: "phish"},
    "model_type": "transformer",
    "exported_at": datetime.now().isoformat()
}

with open(EXPORT_DIR / "dl_meta.json", "w") as f:
    json.dump(dl_export_meta, f, indent=2)

print("✓ Transformer meta saved →", EXPORT_DIR / "dl_meta.json")
print("  (model weights remain in", DL_DIR, ")")

In [5]:
## 4. Unified Inference Class (for the app)

class PhishingClassifier:
    """
    Unified interface for both Classical ML and Transformer models.
    """
    def __init__(self, prefer="transformer"):
        self.prefer = prefer
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Load ML bundle
        self.ml_bundle = joblib.load(EXPORT_DIR / "ml_bundle.joblib")
        self.ml_threshold = self.ml_bundle["threshold"]
        
        # Load Transformer
        # self.tokenizer = AutoTokenizer.from_pretrained(DL_DIR)
        # self.dl_model = AutoModelForSequenceClassification.from_pretrained(DL_DIR).to(self.device)
        # self.dl_model.eval()
        # 
        # with open(EXPORT_DIR / "dl_meta.json") as f:
        #     self.dl_meta = json.load(f)
        # self.dl_threshold = self.dl_meta["threshold"]
        # self.max_length = self.dl_meta["max_length"]
        # 
        # print(f"PhishingClassifier ready (prefer={prefer}, device={self.device})")

    def _predict_ml(self, url: str, text: str):
        # Build feature vector (simplified – in production reuse the exact feature engineering)
        # For demo we only use the text part + dummy numerical features
        # In real app you should re-run the same feature extraction as training
        from sklearn.feature_extraction.text import TfidfVectorizer  # just for structure
        
        # This is a simplified version. In the real app we will re-extract all features.
        # Here we only demonstrate probability from the saved model.
        # (Full feature re-extraction will be in app/inference.py)
        raise NotImplementedError("Full ML feature re-extraction is in app/inference.py")

    def predict_transformer(self, text: str):
        inputs = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)
        
        with torch.no_grad():
            logits = self.dl_model(**inputs).logits
            prob = torch.softmax(logits, dim=-1)[0, 1].item()
        
        label = "phish" if prob >= self.dl_threshold else "benign"
        return {
            "label": label,
            "probability": round(prob, 4),
            "threshold": self.dl_threshold,
            "model": "transformer"
        }

    def predict(self, url: str, html_or_text: str):
        # For now we use the Transformer path (most reliable after export)
        combined = f"{url} [SEP] {html_or_text}"
        return self.predict_transformer(combined[:2000])

In [ ]:
## 5. Quick verification

clf = PhishingClassifier(prefer="transformer")

test_cases = [
    ("https://secure-paypal-login.com/verify", "Enter your password to continue to PayPal"),
    ("https://www.wikipedia.org", "Wikipedia The Free Encyclopedia"),
    ("http://192.168.1.1/login", "Admin panel – please authenticate"),
]

print("\n=== Inference Test ===")
for url, text in test_cases:
    result = clf.predict(url, text)
    print(f"URL: {url}")
    print(f"→ {result}")
    print("-" * 60)

In [ ]:
## 6. Create a simple requirements snippet for the app

req_content = """
# Core
torch
transformers
scikit-learn
xgboost
joblib
pandas
numpy
pyarrow

# App
gradio>=4.0.0
# or fastapi uvicorn for FastAPI version

# Optional
beautifulsoup4
lxml
"""

(EXPORT_DIR / "requirements-app.txt").write_text(req_content.strip())
print("✓ requirements-app.txt written")

In [ ]:
## 7. Final Export Summary

print("\n" + "="*60)
print("EXPORT COMPLETE")
print("="*60)
print(f"Classical ML bundle : {EXPORT_DIR / 'ml_bundle.joblib'}")
print(f"Transformer meta    : {EXPORT_DIR / 'dl_meta.json'}")
print(f"Transformer weights : {DL_DIR}")
print(f"App requirements    : {EXPORT_DIR / 'requirements-app.txt'}")
print("\nYou can now build the Gradio / FastAPI application.")
print("All artefacts needed for inference are ready.")

## Notes for the App

- Use `models/exported/ml_bundle.joblib` for the classical model
- Use `models/dl/best_model` + `models/exported/dl_meta.json` for the Transformer
- The `PhishingClassifier` class above is a starting point – the full production version with proper feature re-extraction will live in `app/inference.py`
- Decision thresholds are already tuned for low false-positive rate

**Next** → Build the Gradio app (`app/main.py`)